1.定义工具,手动通过langchain转化为大模型调用工具

In [14]:

import os

from langchain_core.messages import HumanMessage, ToolMessage
from langchain_core.utils.function_calling import convert_to_openai_tool
from rich import print as rprint

from dotenv import load_dotenv
from langchain_deepseek import ChatDeepSeek

def get_weather(city : str):
    """获取天气工具"""
    return "天气晴朗！！！"

# 加载配置文件，存在相同key采用当前覆盖
load_dotenv(override=True)

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_API_BASE   = os.getenv("DEEPSEEK_BASE_URL")
DEEPSEEK_MODEL_NAME   = os.getenv("DEEPSEEK_MODEL")

deepseek_res = ChatDeepSeek(
    model_name=DEEPSEEK_MODEL_NAME,
    api_key=DEEPSEEK_API_KEY,
    api_base=DEEPSEEK_API_BASE,
)
deepseek_res1 = deepseek_res.bind_tools([get_weather])

message = [
    HumanMessage(content="今天深圳天气如何")
]

res = deepseek_res1.invoke(message)
message.append(res)

tools = res.tool_calls
for tool in tools:
    if tool["name"] == "get_weather":
        too_message = ToolMessage(content=get_weather(tool["args"]),tool_call_id=tool["id"],name=tool["name"])
        message.append(too_message)

for msg in message:
    print(msg)
res1 = deepseek_res1.invoke(message)
# print(convert_to_openai_tool(get_weather))
rprint(res1)

content='今天深圳天气如何' additional_kwargs={} response_metadata={}
content='我来帮您查询深圳今天的天气情况。' additional_kwargs={'refusal': None, 'reasoning_content': 'The user is asking about the weather in Shenzhen today. Let me get the weather for Shenzhen.'} response_metadata={'token_usage': {'completion_tokens': 74, 'prompt_tokens': 353, 'total_tokens': 427, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 21, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 256}, 'prompt_cache_hit_tokens': 256, 'prompt_cache_miss_tokens': 97}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_a18b46594c_prod0820_fp8_kvcache_20260402', 'id': '79474e10-2f36-44fa-9683-e3e9a59c0941', 'finish_reason': 'tool_calls', 'logprobs': None} id='lc_run--019ff441-0368-7173-b4c5-2fde9a624c79-0' tool_calls=[{'name': 'get_weather', 'args': {'city': '深圳'}, 'i

AIMessage(
    content='根据查询结果，**今天深圳天气晴朗**！☀️\n\n是个适合出门的好天气，祝您有愉快的一天！如果您还有其他关于天
气的问题，随时可以问我～',
    additional_kwargs={
        'refusal': None,
        'reasoning_content': 'The tool returned that the weather is sunny. Let me provide this information to the 
user.'
    },
    response_metadata={
        'token_usage': {
            'completion_tokens': 56,
            'prompt_tokens': 442,
            'total_tokens': 498,
            'completion_tokens_details': {
                'accepted_prediction_tokens': None,
                'audio_tokens': None,
                'reasoning_tokens': 18,
                'rejected_prediction_tokens': None
            },
            'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 384},
            'prompt_cache_hit_tokens': 384,
            'prompt_cache_miss_tokens': 58
        },
        'model_provider': 'deepseek',
        'model_name': 'deepseek-v4-flash',
        'system_fingerprint': 'fp_a18b46594c_prod0820_fp8_kvcache_20260402',
        'id': 'bcb11a72-a004-4b60-bb4a-28fd4c8b0bfa',
        'finish_reason': 'stop',
        'logprobs': None
    },
    id='lc_run--019ff441-0813-78b1-9a04-09f02aa93875-0',
    tool_calls=[],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 442,
        'output_tokens': 56,
        'total_tokens': 498,
        'input_token_details': {'cache_read': 384},
        'output_token_details': {'reasoning': 18}
    }
)

1.2 使用@tool注解定义工具，description优先级最高

In [26]:

import os

from langchain.tools import tool

from dotenv import load_dotenv
from langchain_deepseek import ChatDeepSeek

@tool(description="查询具体天气",parse_docstring=True,name_or_callable="getWeather")
def get_weather(city : str):
    """
    获取天气工具

    Args:
        city : 城市
    """
    return "天气晴朗！！！"

# 加载配置文件，存在相同key采用当前覆盖
load_dotenv(override=True)

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_API_BASE   = os.getenv("DEEPSEEK_BASE_URL")
DEEPSEEK_MODEL_NAME   = os.getenv("DEEPSEEK_MODEL")

deepseek_res = ChatDeepSeek(
    model_name=DEEPSEEK_MODEL_NAME,
    api_key=DEEPSEEK_API_KEY,
    api_base=DEEPSEEK_API_BASE,
)
print(convert_to_openai_tool(get_weather))



{'type': 'function', 'function': {'name': 'getWeather', 'description': '查询具体天气', 'parameters': {'properties': {'city': {'description': '城市', 'type': 'string'}}, 'required': ['city'], 'type': 'object'}}}
